# Camada Silver Particionada — Processo Sicredi

Pipeline da camada Silver com **particionamento** por `ano_mes` e **MERGE** (Spark SQL) para deduplicação. Lê as tabelas Delta da camada Bronze (`workspace.bronze`) e grava no schema `workspace.silver` em tabelas com sufixo `_particionado`:

- **Tipagem correta** dos campos (IDs `long`, datas `date`, valores `decimal(18,2)`).
- **Particionamento** por `ano_mes` (formato `yyyy-MM`) extraído da coluna de data de cada entidade.
- **Deduplicação via MERGE**: staging deduplicado com `ROW_NUMBER()` é merged na tabela destino, mantendo apenas a versão mais recente por chave de negócio.
- **Quarentena**: registros inválidos vão para `silver.quarentena_*` com coluna `motivo_erro`.
- **Integridade referencial** validada em cascata: cliente → conta → cartão → transação → estorno.
- **Transações estornadas** identificadas via `flag_estornada` e excluídas de `transacoes_financeiras_particionado`.

In [0]:
# ============================================================
# Parâmetros e criação das tabelas particionadas
# ============================================================
CATALOG = "workspace"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"

# --- DROP e CREATE das tabelas particionadas (schema limpo) ---
spark.sql(f"DROP TABLE IF EXISTS {SILVER}.clientes_particionado")
spark.sql(f"""
CREATE TABLE {SILVER}.clientes_particionado (
  id_cliente        LONG,
  nome              STRING,
  cpf               STRING,
  data_nascimento   DATE,
  cidade            STRING,
  estado            STRING,
  data_atualizacao  DATE,
  motivo_erro       STRING,
  ano_mes           STRING
) USING DELTA
PARTITIONED BY (ano_mes)
""")

spark.sql(f"DROP TABLE IF EXISTS {SILVER}.contas_particionado")
spark.sql(f"""
CREATE TABLE {SILVER}.contas_particionado (
  id_conta       LONG,
  id_cliente     LONG,
  tipo_conta     STRING,
  status_conta   STRING,
  data_abertura  DATE,
  motivo_erro    STRING,
  ano_mes        STRING
) USING DELTA
PARTITIONED BY (ano_mes)
""")

spark.sql(f"DROP TABLE IF EXISTS {SILVER}.cartoes_particionado")
spark.sql(f"""
CREATE TABLE {SILVER}.cartoes_particionado (
  id_cartao        LONG,
  id_conta         LONG,
  bandeira         STRING,
  tipo_cartao      STRING,
  status_cartao    STRING,
  data_emissao     DATE,
  data_atualizacao DATE,
  motivo_erro      STRING,
  ano_mes          STRING
) USING DELTA
PARTITIONED BY (ano_mes)
""")

spark.sql(f"DROP TABLE IF EXISTS {SILVER}.transacoes_cartao_particionado")
spark.sql(f"""
CREATE TABLE {SILVER}.transacoes_cartao_particionado (
  id_transacao           LONG,
  id_cartao              LONG,
  data_transacao         DATE,
  valor                  DECIMAL(18,2),
  nome_estabelecimento   STRING,
  mcc                    INT,
  cidade_estabelecimento STRING,
  flag_estornada         BOOLEAN,
  motivo_erro            STRING,
  ano_mes                STRING
) USING DELTA
PARTITIONED BY (ano_mes)
""")

spark.sql(f"DROP TABLE IF EXISTS {SILVER}.estornos_particionado")
spark.sql(f"""
CREATE TABLE {SILVER}.estornos_particionado (
  id_estorno      LONG,
  id_transacao    LONG,
  data_estorno    DATE,
  motivo_estorno  STRING,
  motivo_erro     STRING,
  ano_mes         STRING
) USING DELTA
PARTITIONED BY (ano_mes)
""")

print("Tabelas particionadas criadas (schema limpo).")

In [0]:
# ============================================================
# 1. CLIENTES — tipagem, validação, quarentena e MERGE dedup
# Particionado por ano_mes (de data_atualizacao).
# Dedup: maior data_atualizacao por id_cliente.
# ============================================================

# 1a. Temp view: tipagem a partir do bronze
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW vw_clientes_tipado AS
SELECT
  CAST(id_cliente AS LONG)          AS id_cliente,
  CAST(nome AS STRING)              AS nome,
  CAST(cpf AS STRING)               AS cpf,
  TRY_CAST(data_nascimento AS DATE)  AS data_nascimento,
  CAST(cidade AS STRING)             AS cidade,
  CAST(estado AS STRING)            AS estado,
  TRY_CAST(data_atualizacao AS DATE) AS data_atualizacao
FROM {BRONZE}.clientes
""")

# 1b. Temp view: validação + ano_mes
spark.sql("""
CREATE OR REPLACE TEMP VIEW vw_clientes_avaliado AS
SELECT
  *,
  CONCAT_WS(' | ',
    CASE WHEN id_cliente IS NULL THEN 'chave_nula:id_cliente' END,
    CASE WHEN NOT cpf RLIKE '^[0-9]{11}$' THEN 'cpf_invalido' END,
    CASE WHEN data_nascimento IS NULL THEN 'data_nascimento_invalida' END,
    CASE WHEN data_nascimento > CURRENT_DATE() THEN 'data_nascimento_futura' END
  ) AS motivo_erro,
  DATE_FORMAT(data_atualizacao, 'yyyy-MM') AS ano_mes
FROM vw_clientes_tipado
""")

# 1c. Quarentena (overwrite)
spark.sql(f"""
CREATE OR REPLACE TABLE {SILVER}.quarentena_clientes AS
SELECT
  CAST(id_cliente AS STRING)       AS id_cliente,
  CAST(nome AS STRING)             AS nome,
  CAST(cpf AS STRING)              AS cpf,
  CAST(data_nascimento AS STRING)  AS data_nascimento,
  CAST(cidade AS STRING)           AS cidade,
  CAST(estado AS STRING)           AS estado,
  CAST(data_atualizacao AS STRING) AS data_atualizacao,
  motivo_erro,
  CURRENT_TIMESTAMP() AS timestamp_processamento
FROM vw_clientes_avaliado
WHERE motivo_erro != ''
""")

# 1d. Staging dedup com ROW_NUMBER (Spark SQL)
spark.sql("""
CREATE OR REPLACE TEMP VIEW vw_clientes_dedup AS
SELECT id_cliente, nome, cpf, data_nascimento, cidade, estado, data_atualizacao, motivo_erro, ano_mes
FROM (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY id_cliente ORDER BY data_atualizacao DESC, cpf DESC) AS _rn
  FROM vw_clientes_avaliado
  WHERE motivo_erro = ''
) WHERE _rn = 1
""")

# 1e. MERGE na tabela particionada
spark.sql(f"""
MERGE INTO {SILVER}.clientes_particionado AS t
USING vw_clientes_dedup AS s
ON t.id_cliente = s.id_cliente
WHEN MATCHED AND s.data_atualizacao > t.data_atualizacao THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

qtd_bronze = spark.table(f"{BRONZE}.clientes").count()
qtd_silver = spark.table(f"{SILVER}.clientes_particionado").count()
qtd_quar   = spark.table(f"{SILVER}.quarentena_clientes").count()
print(f"CLIENTES: bronze={qtd_bronze} | silver_particionado={qtd_silver} | quarentena={qtd_quar}")

In [0]:
# ============================================================
# 2. CONTAS — tipagem, validação, RI e MERGE dedup
# Particionado por ano_mes (de data_abertura).
# RI: id_cliente deve existir em silver.clientes_particionado.
# Dedup: maior data_abertura por id_conta.
# ============================================================

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW vw_contas_tipado AS
SELECT
  CAST(id_conta AS LONG)        AS id_conta,
  CAST(id_cliente AS LONG)      AS id_cliente,
  CAST(tipo_conta AS STRING)    AS tipo_conta,
  CAST(status_conta AS STRING)  AS status_conta,
  TRY_CAST(data_abertura AS DATE) AS data_abertura
FROM {BRONZE}.contas
""")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW vw_contas_avaliado AS
SELECT
  c.*,
  CONCAT_WS(' | ',
    CASE WHEN c.id_conta IS NULL THEN 'chave_nula:id_conta' END,
    CASE WHEN c.id_cliente IS NULL THEN 'chave_nula:id_cliente' END,
    CASE WHEN c.data_abertura IS NULL THEN 'data_abertura_invalida' END,
    CASE WHEN cl.id_cliente IS NULL THEN 'integridade_referencial:id_cliente_inexistente' END
  ) AS motivo_erro,
  DATE_FORMAT(c.data_abertura, 'yyyy-MM') AS ano_mes
FROM vw_contas_tipado c
LEFT JOIN {SILVER}.clientes_particionado cl
  ON c.id_cliente = cl.id_cliente
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {SILVER}.quarentena_contas AS
SELECT
  CAST(id_conta AS STRING)      AS id_conta,
  CAST(id_cliente AS STRING)    AS id_cliente,
  CAST(tipo_conta AS STRING)    AS tipo_conta,
  CAST(status_conta AS STRING)  AS status_conta,
  CAST(data_abertura AS STRING) AS data_abertura,
  motivo_erro,
  CURRENT_TIMESTAMP() AS timestamp_processamento
FROM vw_contas_avaliado
WHERE motivo_erro != ''
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW vw_contas_dedup AS
SELECT id_conta, id_cliente, tipo_conta, status_conta, data_abertura, motivo_erro, ano_mes
FROM (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY id_conta ORDER BY data_abertura DESC, id_cliente DESC) AS _rn
  FROM vw_contas_avaliado
  WHERE motivo_erro = ''
) WHERE _rn = 1
""")

spark.sql(f"""
MERGE INTO {SILVER}.contas_particionado AS t
USING vw_contas_dedup AS s
ON t.id_conta = s.id_conta
WHEN MATCHED AND s.data_abertura > t.data_abertura THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

qtd_bronze = spark.table(f"{BRONZE}.contas").count()
qtd_silver = spark.table(f"{SILVER}.contas_particionado").count()
qtd_quar   = spark.table(f"{SILVER}.quarentena_contas").count()
print(f"CONTAS: bronze={qtd_bronze} | silver_particionado={qtd_silver} | quarentena={qtd_quar}")

In [0]:
# ============================================================
# 3. CARTÕES — tipagem, validação, RI e MERGE dedup
# Particionado por ano_mes (de data_atualizacao).
# RI: id_conta deve existir em silver.contas_particionado.
# Dedup: maior data_atualizacao por id_cartao.
# ============================================================

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW vw_cartoes_tipado AS
SELECT
  CAST(id_cartao AS LONG)          AS id_cartao,
  CAST(id_conta AS LONG)           AS id_conta,
  CAST(bandeira AS STRING)         AS bandeira,
  CAST(tipo_cartao AS STRING)      AS tipo_cartao,
  CAST(status_cartao AS STRING)    AS status_cartao,
  TRY_CAST(data_emissao AS DATE)    AS data_emissao,
  TRY_CAST(data_atualizacao AS DATE) AS data_atualizacao
FROM {BRONZE}.cartoes
""")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW vw_cartoes_avaliado AS
SELECT
  c.*,
  CONCAT_WS(' | ',
    CASE WHEN c.id_cartao IS NULL THEN 'chave_nula:id_cartao' END,
    CASE WHEN c.id_conta IS NULL THEN 'chave_nula:id_conta' END,
    CASE WHEN c.data_emissao IS NULL THEN 'data_emissao_invalida' END,
    CASE WHEN ct.id_conta IS NULL THEN 'integridade_referencial:id_conta_inexistente' END
  ) AS motivo_erro,
  DATE_FORMAT(c.data_atualizacao, 'yyyy-MM') AS ano_mes
FROM vw_cartoes_tipado c
LEFT JOIN {SILVER}.contas_particionado ct
  ON c.id_conta = ct.id_conta
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {SILVER}.quarentena_cartoes AS
SELECT
  CAST(id_cartao AS STRING)        AS id_cartao,
  CAST(id_conta AS STRING)         AS id_conta,
  CAST(bandeira AS STRING)         AS bandeira,
  CAST(tipo_cartao AS STRING)      AS tipo_cartao,
  CAST(status_cartao AS STRING)    AS status_cartao,
  CAST(data_emissao AS STRING)     AS data_emissao,
  CAST(data_atualizacao AS STRING) AS data_atualizacao,
  motivo_erro,
  CURRENT_TIMESTAMP() AS timestamp_processamento
FROM vw_cartoes_avaliado
WHERE motivo_erro != ''
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW vw_cartoes_dedup AS
SELECT id_cartao, id_conta, bandeira, tipo_cartao, status_cartao, data_emissao, data_atualizacao, motivo_erro, ano_mes
FROM (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY id_cartao ORDER BY data_atualizacao DESC, id_conta DESC) AS _rn
  FROM vw_cartoes_avaliado
  WHERE motivo_erro = ''
) WHERE _rn = 1
""")

spark.sql(f"""
MERGE INTO {SILVER}.cartoes_particionado AS t
USING vw_cartoes_dedup AS s
ON t.id_cartao = s.id_cartao
WHEN MATCHED AND s.data_atualizacao > t.data_atualizacao THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

qtd_bronze = spark.table(f"{BRONZE}.cartoes").count()
qtd_silver = spark.table(f"{SILVER}.cartoes_particionado").count()
qtd_quar   = spark.table(f"{SILVER}.quarentena_cartoes").count()
print(f"CARTÕES: bronze={qtd_bronze} | silver_particionado={qtd_silver} | quarentena={qtd_quar}")

In [0]:
# ============================================================
# 4. TRANSAÇÕES — tipagem, validação, RI e MERGE dedup
# Particionado por ano_mes (de data_transacao).
# RI: id_cartao deve existir em silver.cartoes_particionado.
# Dedup: data_transacao/valor por id_transacao (sem data_atualizacao).
# ============================================================

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW vw_transacoes_tipado AS
SELECT
  CAST(id_transacao AS LONG)           AS id_transacao,
  CAST(id_cartao AS LONG)               AS id_cartao,
  TRY_CAST(data_transacao AS DATE)      AS data_transacao,
  TRY_CAST(valor AS DECIMAL(18,2))      AS valor,
  CAST(nome_estabelecimento AS STRING)  AS nome_estabelecimento,
  TRY_CAST(mcc AS INT)                  AS mcc,
  CAST(cidade_estabelecimento AS STRING) AS cidade_estabelecimento
FROM {BRONZE}.transacoes_cartao
""")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW vw_transacoes_avaliado AS
SELECT
  t.*,
  CONCAT_WS(' | ',
    CASE WHEN t.id_transacao IS NULL THEN 'chave_nula:id_transacao' END,
    CASE WHEN t.id_cartao IS NULL THEN 'chave_nula:id_cartao' END,
    CASE WHEN t.data_transacao IS NULL THEN 'data_transacao_invalida' END,
    CASE WHEN t.valor IS NULL OR t.valor <= 0 THEN 'valor_invalido' END,
    CASE WHEN c.id_cartao IS NULL THEN 'integridade_referencial:id_cartao_inexistente' END
  ) AS motivo_erro,
  DATE_FORMAT(t.data_transacao, 'yyyy-MM') AS ano_mes
FROM vw_transacoes_tipado t
LEFT JOIN {SILVER}.cartoes_particionado c
  ON t.id_cartao = c.id_cartao
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {SILVER}.quarentena_transacoes_cartao AS
SELECT
  CAST(id_transacao AS STRING)           AS id_transacao,
  CAST(id_cartao AS STRING)              AS id_cartao,
  CAST(data_transacao AS STRING)         AS data_transacao,
  CAST(valor AS STRING)                  AS valor,
  CAST(nome_estabelecimento AS STRING)   AS nome_estabelecimento,
  CAST(mcc AS STRING)                    AS mcc,
  CAST(cidade_estabelecimento AS STRING) AS cidade_estabelecimento,
  motivo_erro,
  CURRENT_TIMESTAMP() AS timestamp_processamento
FROM vw_transacoes_avaliado
WHERE motivo_erro != ''
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW vw_transacoes_dedup AS
SELECT id_transacao, id_cartao, data_transacao, valor, nome_estabelecimento, mcc, cidade_estabelecimento,
       FALSE AS flag_estornada, motivo_erro, ano_mes
FROM (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY id_transacao ORDER BY data_transacao DESC, valor DESC, id_cartao DESC) AS _rn
  FROM vw_transacoes_avaliado
  WHERE motivo_erro = ''
) WHERE _rn = 1
""")

spark.sql(f"""
MERGE INTO {SILVER}.transacoes_cartao_particionado AS t
USING vw_transacoes_dedup AS s
ON t.id_transacao = s.id_transacao
WHEN MATCHED AND s.data_transacao > t.data_transacao THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

qtd_bronze = spark.table(f"{BRONZE}.transacoes_cartao").count()
qtd_silver = spark.table(f"{SILVER}.transacoes_cartao_particionado").count()
qtd_quar   = spark.table(f"{SILVER}.quarentena_transacoes_cartao").count()
print(f"TRANSAÇÕES: bronze={qtd_bronze} | silver_particionado={qtd_silver} | quarentena={qtd_quar}")

In [0]:
# ============================================================
# 5. ESTORNOS — tipagem, validação, RI e MERGE dedup
# Particionado por ano_mes (de data_estorno).
# RI: id_transacao deve existir em silver.transacoes_cartao_particionado.
# ============================================================

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW vw_estornos_tipado AS
SELECT
  CAST(id_estorno AS LONG)        AS id_estorno,
  CAST(id_transacao AS LONG)      AS id_transacao,
  TRY_CAST(data_estorno AS DATE)  AS data_estorno,
  CAST(motivo_estorno AS STRING)  AS motivo_estorno
FROM {BRONZE}.estornos
""")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW vw_estornos_avaliado AS
SELECT
  e.*,
  CONCAT_WS(' | ',
    CASE WHEN e.id_estorno IS NULL THEN 'chave_nula:id_estorno' END,
    CASE WHEN e.id_transacao IS NULL THEN 'chave_nula:id_transacao' END,
    CASE WHEN e.data_estorno IS NULL THEN 'data_estorno_invalida' END,
    CASE WHEN t.id_transacao IS NULL THEN 'integridade_referencial:id_transacao_inexistente' END
  ) AS motivo_erro,
  DATE_FORMAT(e.data_estorno, 'yyyy-MM') AS ano_mes
FROM vw_estornos_tipado e
LEFT JOIN {SILVER}.transacoes_cartao_particionado t
  ON e.id_transacao = t.id_transacao
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {SILVER}.quarentena_estornos AS
SELECT
  CAST(id_estorno AS STRING)      AS id_estorno,
  CAST(id_transacao AS STRING)    AS id_transacao,
  CAST(data_estorno AS STRING)    AS data_estorno,
  CAST(motivo_estorno AS STRING)  AS motivo_estorno,
  motivo_erro,
  CURRENT_TIMESTAMP() AS timestamp_processamento
FROM vw_estornos_avaliado
WHERE motivo_erro != ''
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW vw_estornos_dedup AS
SELECT id_estorno, id_transacao, data_estorno, motivo_estorno, motivo_erro, ano_mes
FROM (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY id_estorno ORDER BY data_estorno DESC, id_transacao DESC) AS _rn
  FROM vw_estornos_avaliado
  WHERE motivo_erro = ''
) WHERE _rn = 1
""")

spark.sql(f"""
MERGE INTO {SILVER}.estornos_particionado AS t
USING vw_estornos_dedup AS s
ON t.id_estorno = s.id_estorno
WHEN MATCHED AND s.data_estorno > t.data_estorno THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

qtd_bronze = spark.table(f"{BRONZE}.estornos").count()
qtd_silver = spark.table(f"{SILVER}.estornos_particionado").count()
qtd_quar   = spark.table(f"{SILVER}.quarentena_estornos").count()
print(f"ESTORNOS: bronze={qtd_bronze} | silver_particionado={qtd_silver} | quarentena={qtd_quar}")

# ============================================================
# 6. FLAG DE ESTORNO via MERGE + TRANSÇÕES FINANCEIRAS
# ============================================================

# Marca flag_estornada = TRUE nas transações que possuem estorno
spark.sql(f"""
MERGE INTO {SILVER}.transacoes_cartao_particionado AS t
USING (
  SELECT DISTINCT id_transacao
  FROM {SILVER}.estornos_particionado
) AS s
ON t.id_transacao = s.id_transacao
WHEN MATCHED THEN UPDATE SET t.flag_estornada = TRUE
""")

# Cria transacoes_financeiras_particionado (apenas transações NÃO estornadas)
spark.sql(f"""
CREATE OR REPLACE TABLE {SILVER}.transacoes_financeiras_particionado
USING DELTA
PARTITIONED BY (ano_mes)
AS SELECT
  id_transacao, id_cartao, data_transacao, valor, nome_estabelecimento, mcc,
  cidade_estabelecimento, flag_estornada, motivo_erro, ano_mes
FROM {SILVER}.transacoes_cartao_particionado
WHERE COALESCE(flag_estornada, FALSE) = FALSE
""")

total_trans = spark.table(f"{SILVER}.transacoes_cartao_particionado").count()
total_estornadas = spark.table(f"{SILVER}.transacoes_cartao_particionado").filter("flag_estornada = true").count()
total_financeiras = spark.table(f"{SILVER}.transacoes_financeiras_particionado").count()
print(f"Transações: total={total_trans} | estornadas={total_estornadas} | financeiras (sem estorno)={total_financeiras}")

In [0]:
# ============================================================
# 7. RESUMO DO PIPELINE SILVER PARTICIONADO
# ============================================================
from pyspark.sql import functions as F

resumo = [
    ("clientes", f"{BRONZE}.clientes", f"{SILVER}.clientes_particionado", f"{SILVER}.quarentena_clientes"),
    ("contas", f"{BRONZE}.contas", f"{SILVER}.contas_particionado", f"{SILVER}.quarentena_contas"),
    ("cartoes", f"{BRONZE}.cartoes", f"{SILVER}.cartoes_particionado", f"{SILVER}.quarentena_cartoes"),
    ("transacoes_cartao", f"{BRONZE}.transacoes_cartao", f"{SILVER}.transacoes_cartao_particionado", f"{SILVER}.quarentena_transacoes_cartao"),
    ("estornos", f"{BRONZE}.estornos", f"{SILVER}.estornos_particionado", f"{SILVER}.quarentena_estornos"),
]

linhas = []
for entidade, t_bronze, t_silver, t_quar in resumo:
    bronze_n = spark.table(t_bronze).count()
    silver_n = spark.table(t_silver).count()
    quar_n = spark.table(t_quar).count()
    pct = round(100.0 * quar_n / bronze_n, 2) if bronze_n else 0.0
    linhas.append((entidade, bronze_n, silver_n, quar_n, pct))

df_resumo = spark.createDataFrame(
    linhas,
    "entidade string, bronze long, silver_particionado long, quarentena long, pct_quarentena double",
)
display(df_resumo)

# Distribuição dos motivos de erro nas tabelas de quarentena
motivos = []
for entidade, _, _, t_quar in resumo:
    m = (
        spark.table(t_quar)
        .groupBy("motivo_erro")
        .count()
        .withColumn("entidade", F.lit(entidade))
        .select("entidade", "motivo_erro", "count")
    )
    motivos.append(m)

df_motivos = motivos[0]
for m in motivos[1:]:
    df_motivos = df_motivos.unionByName(m)

display(df_motivos.orderBy("entidade", F.col("count").desc()))

# Partições por tabela
print("\nPartições (ano_mes) por tabela:")
for entidade, _, t_silver, _ in resumo:
    partes = spark.sql(f"SHOW PARTITIONS {t_silver}").count()
    print(f"  {entidade}: {partes} partições")